# Recommender Systems - Part A Project
**Paper:** SVD-GoRank: Recommender System Algorithm Using SVD and Gower's Ranking

**Dataset:** MovieLens-10M

**Team Members:** Harun Korkmaz, Muhammet Salih Hasılcıo, Orhan Efe Bayrak, Muhiddin Fırat, Yasin Furkan Abasız, Yakup Berkay Genceroğlu, Görkem Yahya Bakan

This notebook contains step-by-step implementation of the SVD-GoRank algorithm on the MovieLens-10M dataset containing 10 million ratings.

## PROGRESS 1: Data Loading & Preprocessing
In this step, the original MovieLens-10M dataset is loaded into the system using Pandas, unnecessary columns (timestamp) are removed, and statistical properties mentioned in the paper (especially Sparsity rate) are calculated.

In [1]:
import pandas as pd
import numpy as np

# 1. Load the MovieLens-10M dataset
# Assume the 'ratings.dat' file extracted from the zip is in the same directory
# Use engine='python' because separator is '::'
file_path = r"C:\Users\Harun\Desktop\üni\4.SINIF\2.donem\recommender\dataset\ml-10M100K\ratings.dat" 
columns = ['user_id', 'item_id', 'rating', 'timestamp']

# Reading this data may take a few seconds as it is very large
df_10m = pd.read_csv(file_path, sep='::', names=columns, engine='python')

# Drop timestamp column
df_10m = df_10m.drop('timestamp', axis=1)

# Calculate statistics
n_users = df_10m['user_id'].nunique()
n_items = df_10m['item_id'].nunique()
n_ratings = len(df_10m)

# Sparsity Formula
sparsity = 1.0 - (n_ratings / (n_users * n_items))

print("--- MOVIELENS 10M - PROGRESS 1 ---")
print(f"Total Users: {n_users}")
print(f"Total Movies: {n_items}")
print(f"Total Ratings: {n_ratings}")
print(f"Sparsity Ratio: {sparsity:.4f} ({sparsity*100:.2f}%)")
print("\nFirst 5 Rows of Dataset:")
display(df_10m.head())

--- MOVIELENS 10M - PROGRESS 1 ---
Total Users: 69878
Total Movies: 10677
Total Ratings: 10000054
Sparsity Ratio: 0.9866 (98.66%)

First 5 Rows of Dataset:


,user_id,item_id,rating
0,1,122,5.0
1,1,185,5.0
2,1,231,5.0
3,1,292,5.0
4,1,316,5.0


## PROGRESS 2: Train/Test Split & User-Item Matrix Creation
To fairly evaluate the model, we split the data into 80% training and 20% test sets. Then, we construct the "User-Item Interaction Matrix" which will be used in Phase-1 (SVD) of the paper and is currently mostly empty (NaN).

In [2]:
from sklearn.model_selection import train_test_split

# 1. Split data into 80% Training and 20% Testing
train_data_10m, test_data_10m = train_test_split(df_10m, test_size=0.20, random_state=42)

print("--- MOVIELENS 10M - PROGRESS 2 ---")
print(f"Training Set Size: {len(train_data_10m)} rows")
print(f"Test Set Size: {len(test_data_10m)} rows\n")

print("ATTENTION - ENGINEERING NOTE:")
print("Creating a 70,000 x 10,000 Pandas Pivot table at this stage may consume approximately 5-6 GB RAM.")
print("If the system raises 'MemoryError', we will switch to 'scipy.sparse' sparse matrix")
print("structure when implementing the SVD algorithm.\n")

# 2. Build User-Item Matrix from training data
# If computer freezes, comment out this line and use Sparse Matrix during SVD phase
user_item_matrix_10m = train_data_10m.pivot(index='user_id', columns='item_id', values='rating')

print(f"User-Item Matrix Size: {user_item_matrix_10m.shape}\n")
print("Matrix Visualization (Empty cells are NaN):")
display(user_item_matrix_10m.head())

--- MOVIELENS 10M - PROGRESS 2 ---
Training Set Size: 8000043 rows
Test Set Size: 2000011 rows

ATTENTION - ENGINEERING NOTE:
Creating a 70,000 x 10,000 Pandas Pivot table at this stage may consume approximately 5-6 GB RAM.
If the system raises 'MemoryError', we will switch to 'scipy.sparse' sparse matrix
structure when implementing the SVD algorithm.

User-Item Matrix Size: (69878, 10653)

Matrix Visualization (Empty cells are NaN):


item_id,1,2,3,4,5,6,7,8,9,10,...,65006,65011,65025,65027,65037,65088,65091,65126,65130,65133
user_id,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1.0,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
